# Auditoria das previsões do champion

Compara `review_data/` (previsões originais) com `curated_data/` (anotações humanas) usando scores dos sidecars em `production_predictions/` e `COCOeval`. A mesma implementação é usada por `run_curation.sh` antes de decidir se deve disparar o retreinamento.

O campo `Precision` abaixo reproduz a saída anterior do notebook (`evaluator.stats[0]`), portanto é igual ao mAP@0.50:0.95 e não é uma medida independente de precisão.

## 1. Carregar a avaliação compartilhada

In [ ]:
from src.data_automation.curation_metrics import (
    RETRAIN_THRESHOLD,
    evaluate_curation,
)


## 2. Converter os dados e executar COCOeval

In [ ]:
evaluation = evaluate_curation()


## 3. Métricas e decisão do gate

In [ ]:
print(f"Execução: {evaluation.run_id}")
print(f"Imagens: {evaluation.image_count}")
print(f"Predições com score: {evaluation.prediction_count}")
print(f"Caixas humanas: {evaluation.ground_truth_count}")
print(f"Caixas degeneradas incluídas: {evaluation.degenerate_prediction_count}")
if evaluation.metrics is None:
    print("Métricas COCO indefinidas: não há caixas humanas.")
    should_retrain = False
else:
    for name, value in evaluation.metrics.as_dict().items():
        print(f"{name}: {value}")
    should_retrain = evaluation.metrics.all_below(RETRAIN_THRESHOLD)
print(f"RETRAIN_REQUIRED={str(should_retrain).lower()}")
